# Crack-detection experiments on Colab

Runs the baseline and analysis experiments for the AG-DSCAE paper.

Before starting: **Runtime -> Change runtime type -> GPU** (T4 is enough for
PatchCore/PaDiM; DRAEM/EfficientAD train slower but fit).

Datasets are read from Google Drive and all outputs are written back to
Drive, so progress survives Colab disconnects.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Alirezanltv/Unsupervised-Crack-Detection_CVPR.git
%cd Unsupervised-Crack-Detection_CVPR
# torch/torchvision come preinstalled on Colab; install the rest
!pip install -q anomalib scikit-image scikit-learn opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Expected Drive layout

```
MyDrive/crack_data/
  concrete/  road/  bridge/     each with:
    train/good/    unlabeled training images
    test/images/   test images
    test/masks/    binary masks, same file stems
    calib/         held-out unlabeled images
```
Adjust `DATA`/`RUNS` below if your paths differ.

## Download datasets directly (no upload needed)

Verified direct links (run once; everything lands on Drive so later
sessions skip this). The Ozgenel classification set comes via the Kaggle
CLI: create an API token (kaggle.com -> Settings -> Create New Token) and
upload the kaggle.json it gives you when prompted.

Note: these are the RAW public datasets. Reproducing the paper's exact
train/val/test splits and the 1,842-image bridge-deck subset requires the
curation lists from the original experiments.

In [ ]:
RAW = '/content/drive/MyDrive/crack_raw'
import os; os.makedirs(RAW, exist_ok=True)
%cd $RAW

# SDNET2018 (bridge decks/walls/pavements; 528 MB) -- Utah State University
!wget -c -O sdnet2018.zip 'https://digitalcommons.usu.edu/context/all_datasets/article/1047/type/native/viewcontent'
!unzip -q -n sdnet2018.zip -d sdnet2018

# DeepCrack benchmark (images + masks, official split)
!wget -c https://raw.githubusercontent.com/yhlleo/DeepCrack/master/dataset/DeepCrack.zip
!unzip -q -n DeepCrack.zip -d deepcrack

# CrackForest (CFD, 118 road images with annotations)
!git clone --depth 1 https://github.com/cuilimeng/CrackForest-dataset.git crackforest

# BSDS500 (natural images for the Canny/HED pretraining control)
!wget -c https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz
!tar xzf BSR_bsds500.tgz

In [ ]:
# Ozgenel 40k concrete classification set via its Kaggle mirror
# (Mendeley's direct links are login-gated; the mirror is scriptable).
!pip -q install kaggle
from google.colab import files
print('upload your kaggle.json'); files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d arunrk7/surface-crack-detection -p $RAW/ozgenel --unzip
# if the slug ever moves: search Kaggle for 'Surface Crack Detection' (40k, 227x227)

In [ ]:
DATA = '/content/drive/MyDrive/crack_data'
RUNS = '/content/drive/MyDrive/crack_runs'
import os; os.makedirs(RUNS, exist_ok=True)

In [ ]:
# sanity checks, no data needed (~1 min)
!python common/stats.py --selftest
!python common/smoke_test.py

In [ ]:
# MNIST upsampling analysis (CPU, ~5 min) -- should reproduce the committed
# results.json / results_schemes.json
!python p3_upsampling_stats/edge_stats.py --n 1000 --n-profile 150
!python p3_upsampling_stats/edge_stats.py --n 1000 --n-profile 150 --schemes

## Baselines

Start small to validate the pipeline end to end: two fast memory-bank
models, one dataset, one seed. Then widen to all models/datasets/seeds
(edit the flags). One-class training data choice is discussed in the
script header -- decide and note which option you used.

In [ ]:
!python p1_sota_baselines/run_baselines.py \
    --data $DATA/concrete --out $RUNS/concrete \
    --models patchcore padim --seeds 0

In [ ]:
# score a finished run (repeat per model/seed)
!python common/eval_maps.py \
    --maps $RUNS/concrete/patchcore/s0/maps \
    --masks $DATA/concrete/test/masks \
    --calib $RUNS/concrete/patchcore/s0/calib \
    --out $RUNS/concrete/patchcore/s0/result.json

## Scaling up + what to report back

- Full sweep: `--models patchcore padim draem rd4ad efficientad --seeds 0 1 2 3 4`,
  once per dataset. Long jobs: run one model per session; outputs land on
  Drive, so a disconnect only loses the model currently training.
- Our model's maps: dump raw anomaly maps as .npy (same stems) plus calib
  maps from the trainer, then score them with the same eval_maps.py call --
  this yields the AUROC/AP columns for the existing tables.
- Attention gates: upload a checkpoint, adapt load_model_and_gates() in
  p3_attention_viz/visualize_gates.py, and run it on test/images.

Collect and share: every `result.json`, `gate_stats.csv` + `gate_maps.png`,
and the per-seed MIoU table for stats.py.